# TT

In [ ]:
import torch
import torch.nn as nn
import tltorch

## Загрузка модели

In [2]:
def analyze_model_parameters(model):
    print(f"{'Layer (Module)':<35} {'Output Shape':<25} {'Param #':<15}")
    print("=" * 90)

    total_params = 0
    trainable_params = 0

    for name, param in model.named_parameters():
        param_count = param.numel()
        
        total_params += param_count
        if param.requires_grad:
            trainable_params += param_count
        
        print(f"{name:<35} {str(list(param.shape)):<25} {param_count:<15,}")

    print("=" * 90)
    print(f"Total parameters: {total_params:,}")

In [3]:
from pytorch.imp.model import SS_ConvNeXt_HSI_V2

In [4]:
model = SS_ConvNeXt_HSI_V2()
checkpoint = torch.load("best_ssv2.pth", weights_only=False)
model.load_state_dict(checkpoint['state_dict'])

<All keys matched successfully>

## Анализ модели

In [6]:
analyze_model_parameters(model)

Layer (Module)                      Output Shape              Param #        
downsample_layers.0.0.weight        [64, 204, 1, 1]           13,056         
downsample_layers.0.0.bias          [64]                      64             
downsample_layers.0.1.weight        [64]                      64             
downsample_layers.0.1.bias          [64]                      64             
downsample_layers.1.0.weight        [64]                      64             
downsample_layers.1.0.bias          [64]                      64             
downsample_layers.1.1.weight        [128, 64, 2, 2]           32,768         
downsample_layers.1.1.bias          [128]                     128            
downsample_layers.2.0.weight        [128]                     128            
downsample_layers.2.0.bias          [128]                     128            
downsample_layers.2.1.weight        [256, 128, 2, 2]          131,072        
downsample_layers.2.1.bias          [256]                     25

## Сжатие модели

In [7]:
def replace_linear_with_tt_transformation(module, layer_name, rank):
    """
    Заменяет nn.Linear на tltorch.FactorizedLinear с TT-разложением
    """
    old_layer = getattr(module, layer_name)
    
    if not isinstance(old_layer, nn.Linear):
        return

    print(f"Compressing Linear layer: {layer_name} (shape: {old_layer.weight.shape}) with TT-rank {rank}")

    try:
        tt_layer = tltorch.FactorizedLinear.from_linear(old_layer, rank=rank, factorization='blocktt', implementation='factorized')

        setattr(module, layer_name, tt_layer)
        
        print(f"   -> Success.")
        
    except Exception as e:
        print(f"   -> Error during TT conversion: {e}")

In [8]:
def replace_linear_with_tt_transformation(module, layer_name, rank):
    """
    Заменяет nn.Linear на tltorch.FactorizedLinear с TT-разложением
    """
    old_layer = getattr(module, layer_name)
    
    if not isinstance(old_layer, nn.Linear):
        return

    print(f"Compressing Linear layer: {layer_name} (shape: {old_layer.weight.shape}) with TT-rank {rank}")

    try:
        tt_layer = tltorch.FactorizedLinear.from_linear(old_layer, rank=rank, factorization='blocktt', implementation='factorized')

        setattr(module, layer_name, tt_layer)
        
        print(f"Success.")
        
    except Exception as e:
        print(f"Error during TT conversion: {e}")

def change_convnext_TT(model: nn.Module, rank_list=[1.0, 0.75, 0.5, 0.3]):

    rank_iter = iter(rank_list)

    for block_idx in range(len(model.main_layers)):

        r = next(rank_iter)
        
        for sub_idx in range(len(model.main_layers[block_idx])):
            
            try:
                replace_linear_with_tt_transformation(model.main_layers[block_idx][sub_idx], 'pwconv1', rank=r)
                replace_linear_with_tt_transformation(model.main_layers[block_idx][sub_idx], 'pwconv2', rank=r)
            except AttributeError:
                continue
            except IndexError:
                continue

In [9]:
change_convnext_TT(model)

Compressing Linear layer: pwconv1 (shape: torch.Size([256, 64])) with TT-rank 1.0
Success.
Compressing Linear layer: pwconv2 (shape: torch.Size([64, 256])) with TT-rank 1.0
Success.
Compressing Linear layer: pwconv1 (shape: torch.Size([256, 64])) with TT-rank 1.0
Success.
Compressing Linear layer: pwconv2 (shape: torch.Size([64, 256])) with TT-rank 1.0
Success.
Compressing Linear layer: pwconv1 (shape: torch.Size([512, 128])) with TT-rank 0.75
Success.
Compressing Linear layer: pwconv2 (shape: torch.Size([128, 512])) with TT-rank 0.75
Success.
Compressing Linear layer: pwconv1 (shape: torch.Size([512, 128])) with TT-rank 0.75
Success.
Compressing Linear layer: pwconv2 (shape: torch.Size([128, 512])) with TT-rank 0.75
Success.
Compressing Linear layer: pwconv1 (shape: torch.Size([1024, 256])) with TT-rank 0.5
Success.
Compressing Linear layer: pwconv2 (shape: torch.Size([256, 1024])) with TT-rank 0.5
Success.
Compressing Linear layer: pwconv1 (shape: torch.Size([1024, 256])) with TT-ran

In [10]:
# start_value = 0.5
# multiplier = 0.95
# rank_list = [start_value]

# for i in range(len(model.main_layers)):
#     for j in range(len(model.main_layers[i])):
#         rank_list.append(multiplier*rank_list[-1])

## Анализ сжатия

In [11]:
model.main_layers[0][0].pwconv1

FactorizedLinear(in_features=64, out_features=256, weight of size (256, 64) tensorized to ((4, 4, 16), (4, 4, 4)),factorization=BlockTT, rank=[1, 18, 45, 1], implementation=factorized, with a single layer parametrized, 

In [12]:
analyze_model_parameters(model)

Layer (Module)                      Output Shape              Param #        
downsample_layers.0.0.weight        [64, 204, 1, 1]           13,056         
downsample_layers.0.0.bias          [64]                      64             
downsample_layers.0.1.weight        [64]                      64             
downsample_layers.0.1.bias          [64]                      64             
downsample_layers.1.0.weight        [64]                      64             
downsample_layers.1.0.bias          [64]                      64             
downsample_layers.1.1.weight        [128, 64, 2, 2]           32,768         
downsample_layers.1.1.bias          [128]                     128            
downsample_layers.2.0.weight        [128]                     128            
downsample_layers.2.0.bias          [128]                     128            
downsample_layers.2.1.weight        [256, 128, 2, 2]          131,072        
downsample_layers.2.1.bias          [256]                     25

In [13]:
arch = type(model).__name__
state = {
    'arch': arch+"_TT",
    'state_dict': model.state_dict(),
}
filename = str('best_ssv2_tt.pth')
torch.save(state, filename)

## Тест сжатия

In [14]:
from pytorch.imp.dataloader_test import test_dataloader
import pytorch.imp.metric as module_metric

In [15]:
loader = test_dataloader

data_iter = iter(loader)

In [16]:
model.to('cuda')

SS_ConvNeXt_V2(
  (downsample_layers): ModuleList(
    (0): Sequential(
      (0): Conv2d(204, 64, kernel_size=(1, 1), stride=(1, 1))
      (1): LayerNorm()
      (2): GELU(approximate='none')
    )
    (1): Sequential(
      (0): LayerNorm()
      (1): Conv2d(64, 128, kernel_size=(2, 2), stride=(2, 2))
    )
    (2): Sequential(
      (0): LayerNorm()
      (1): Conv2d(128, 256, kernel_size=(2, 2), stride=(2, 2))
    )
    (3): Sequential(
      (0): LayerNorm()
      (1): Conv2d(256, 512, kernel_size=(2, 2), stride=(2, 2))
    )
  )
  (main_layers): ModuleList(
    (0): Sequential(
      (0): spatial_ConvBlock(
        (dwconv): Conv2d(64, 64, kernel_size=(3, 11), stride=(1, 1), padding=same, groups=64)
        (norm): LayerNorm()
        (pwconv1): FactorizedLinear(in_features=64, out_features=256, weight of size (256, 64) tensorized to ((4, 4, 16), (4, 4, 4)),factorization=BlockTT, rank=[1, 18, 45, 1], implementation=factorized, with a single layer parametrized, 
        (act): GEL

In [17]:
next_batch = next(data_iter)
next_batch = [_.to('cuda', non_blocking=True) for _ in next_batch]
(data, target) = next_batch 
output = model(data)
module_metric.OA('cuda', output=output, target=target)

tensor(0.7500, device='cuda:0')

In [18]:
checkpoint_tt = torch.load("best_ssv2_tt.pth", weights_only = False)

In [19]:
model.load_state_dict(checkpoint_tt['state_dict'])

<All keys matched successfully>

In [30]:
output = model(data)
module_metric.OA('cuda', output=output, target=target)

tensor(0.6250, device='cuda:0')

# Pruning

In [1]:
import torch

## Загрузка модели

In [2]:
import pytorch.imp.model as module_arch

In [3]:
model = module_arch.SS_ConvNeXt_HSI_V2()

In [4]:
checkpoint = torch.load("best_ssv2.pth", weights_only=False)
model.load_state_dict(checkpoint['state_dict'])

<All keys matched successfully>

## Анализ модели

In [5]:
def analyze_model_parameters(model):
    print(f"{'Layer (Module)':<35} {'Output Shape':<25} {'Param #':<15}")
    print("=" * 90)

    total_params = 0
    trainable_params = 0

    for name, param in model.named_parameters():
        param_count = param.numel()
        
        total_params += param_count
        if param.requires_grad:
            trainable_params += param_count
        
        print(f"{name:<35} {str(list(param.shape)):<25} {param_count:<15,}")

    print("=" * 90)
    print(f"Total parameters: {total_params:,}")

In [6]:
analyze_model_parameters(model)

Layer (Module)                      Output Shape              Param #        
downsample_layers.0.0.weight        [64, 204, 1, 1]           13,056         
downsample_layers.0.0.bias          [64]                      64             
downsample_layers.0.1.weight        [64]                      64             
downsample_layers.0.1.bias          [64]                      64             
downsample_layers.1.0.weight        [64]                      64             
downsample_layers.1.0.bias          [64]                      64             
downsample_layers.1.1.weight        [128, 64, 2, 2]           32,768         
downsample_layers.1.1.bias          [128]                     128            
downsample_layers.2.0.weight        [128]                     128            
downsample_layers.2.0.bias          [128]                     128            
downsample_layers.2.1.weight        [256, 128, 2, 2]          131,072        
downsample_layers.2.1.bias          [256]                     25

## Сжатие модели

In [7]:
module_arch.prune_convnext(model=model)

## Анализ сжатия

In [9]:
analyze_model_parameters(model)

Layer (Module)                      Output Shape              Param #        
downsample_layers.0.0.weight        [64, 204, 1, 1]           13,056         
downsample_layers.0.0.bias          [64]                      64             
downsample_layers.0.1.weight        [64]                      64             
downsample_layers.0.1.bias          [64]                      64             
downsample_layers.1.0.weight        [64]                      64             
downsample_layers.1.0.bias          [64]                      64             
downsample_layers.1.1.weight        [128, 64, 2, 2]           32,768         
downsample_layers.1.1.bias          [128]                     128            
downsample_layers.2.0.weight        [128]                     128            
downsample_layers.2.0.bias          [128]                     128            
downsample_layers.2.1.weight        [256, 128, 2, 2]          131,072        
downsample_layers.2.1.bias          [256]                     25

In [10]:
arch = type(model).__name__
state = {
    'arch': arch,
    'state_dict': model.state_dict(),
}
filename = str('best_ssv2_struct_pruned.pth')
torch.save(state, filename)

## Тест сжатия

In [11]:
from pytorch.imp.dataloader_test import test_dataloader
import pytorch.imp.metric as module_metric

In [12]:
loader = test_dataloader

data_iter = iter(loader)

In [13]:
model.to('cuda')

SS_ConvNeXt_V2(
  (downsample_layers): ModuleList(
    (0): Sequential(
      (0): Conv2d(204, 64, kernel_size=(1, 1), stride=(1, 1))
      (1): LayerNorm()
      (2): GELU(approximate='none')
    )
    (1): Sequential(
      (0): LayerNorm()
      (1): Conv2d(64, 128, kernel_size=(2, 2), stride=(2, 2))
    )
    (2): Sequential(
      (0): LayerNorm()
      (1): Conv2d(128, 256, kernel_size=(2, 2), stride=(2, 2))
    )
    (3): Sequential(
      (0): LayerNorm()
      (1): Conv2d(256, 512, kernel_size=(2, 2), stride=(2, 2))
    )
  )
  (main_layers): ModuleList(
    (0): Sequential(
      (0): spatial_ConvBlock(
        (dwconv): Conv2d(64, 64, kernel_size=(3, 11), stride=(1, 1), padding=same, groups=64)
        (norm): LayerNorm()
        (pwconv1): Linear(in_features=64, out_features=256, bias=True)
        (act): GELU(approximate='none')
        (grn): GRN()
        (pwconv2): Linear(in_features=256, out_features=64, bias=True)
        (drop_path): Identity()
      )
      (1): spe

In [14]:
next_batch = next(data_iter)
next_batch = [_.to('cuda', non_blocking=True) for _ in next_batch]
(data, target) = next_batch 
output = model(data)
module_metric.OA('cuda', output=output, target=target)

tensor(0.9375, device='cuda:0')

In [15]:
checkpoint_pruned = torch.load("best_ssv2_struct_pruned.pth", weights_only = False)

In [16]:
model.load_state_dict(checkpoint_pruned['state_dict'])

<All keys matched successfully>

In [33]:
output = model(data)
module_metric.OA('cuda', output=output, target=target)

tensor(0.9375, device='cuda:0')